# Model Training Pipeline

## Data Preparation

### PDF Content Extraction

We have a PDF autobiography of Jesse Livermore called *Reminiscences of a Stock Operator* here.

Given that the PDF has chapters labeled already, let's keep only chapters and remove unnecessary parts.

In [1]:
from PyPDF2 import PdfReader

# Load the PDF file
pdf_path = "Reminiscences of a Stock Operator 2008.pdf"
reader = PdfReader(pdf_path)

outlines = reader.outline

In [2]:
def print_chapter_titles(outlines):
    for item in outlines:
        if isinstance(item, list):
            print_chapter_titles(item)  # handle nested outlines
        else:
            print(item.title)

print_chapter_titles(outlines)

Front
Title
Copyright
Dedication
Index
I
II
III
IV
V
VI
VII
VIII
IX
X
XI
XII
XIII
XIV
XV
XVI
XVII
XVIII
XIX
XX
XXI
XXII
XXIII
XXIV
Back


In [3]:
chapter_info = []
for item in outlines:
    title = item.title
    page_number = reader.get_destination_page_number(item)
    chapter_info.append((title, page_number))
print(chapter_info)

[('Front\r', 0), ('Title', 2), ('Copyright', 3), ('Dedication', 4), ('Index', 6), ('I', 8), ('II', 22), ('III', 38), ('IV', 48), ('V', 66), ('VI', 80), ('VII', 92), ('VIII', 100), ('IX', 114), ('X', 132), ('XI', 146), ('XII', 160), ('XIII', 178), ('XIV', 192), ('XV', 210), ('XVI', 220), ('XVII', 238), ('XVIII', 254), ('XIX', 264), ('XX', 278), ('XXI', 286), ('XXII', 304), ('XXIII', 326), ('XXIV', 338), ('Back\r', 347)]


In [4]:
chapter_info = chapter_info[5:-1]

In [5]:
chapter_info

[('I', 8),
 ('II', 22),
 ('III', 38),
 ('IV', 48),
 ('V', 66),
 ('VI', 80),
 ('VII', 92),
 ('VIII', 100),
 ('IX', 114),
 ('X', 132),
 ('XI', 146),
 ('XII', 160),
 ('XIII', 178),
 ('XIV', 192),
 ('XV', 210),
 ('XVI', 220),
 ('XVII', 238),
 ('XVIII', 254),
 ('XIX', 264),
 ('XX', 278),
 ('XXI', 286),
 ('XXII', 304),
 ('XXIII', 326),
 ('XXIV', 338)]

Let's store the chapters.

In [12]:
chapters = []

for idx, (title, start_page) in enumerate(chapter_info):
    end_page = chapter_info[idx + 1][1] if idx + 1 < len(chapter_info) else len(reader.pages)
    
    chapter_text = ""
    for p in range(start_page, end_page):
        chapter_text += reader.pages[p].extract_text() + "\n"
    chapters += [chapter_text]
    
    print(f"=== {title} ===")
    print(chapter_text[:500])  # Print the first 500 characters of chapter
    print("\n\n")

=== I ===
I
I went to work when I was just out of grammar school. I  
got  a  job  as  quotation-board  boy  in  a  stock-brokerage  
office. I was quick at figures. At school I did three years of  
arithmetic  in  one.  I  was  particularly  good  at  mental  
arithmetic. As quotation-board boy I posted the numbers  
on  the  big  board  in  the  customers'  room.  One  of  the  
customers  usually  sat  by  the  ticker  and  called  out  the  
prices. They couldn't come too fast for me. I have always  




=== II ===
II
Between  the  discovery  that  the  Cosmopolitan  Stock  
Brokerage Company was ready to beat me by foul means if  
the killing handicap of a three-point margin and a point-
and-a-half premium didn't do it, and hints that they didn't  
want my business anyhow, I soon  made up my mind to go  
to New York, where I could trade in the office of some  
member of the New York Stock Exchange. I didn't want  
any  Boston  branch,  where  the  quotations  had  to  be  
telegra

In [19]:
for i in range(1, len(chapters) + 1):
    print("Chapter ", i, ": ", end="", sep="")
    print(len(chapters[i - 1]))

Chapter 1: 26540
Chapter 2: 29821
Chapter 3: 18192
Chapter 4: 32266
Chapter 5: 25142
Chapter 6: 19794
Chapter 7: 13398
Chapter 8: 27079
Chapter 9: 34163
Chapter 10: 28749
Chapter 11: 24862
Chapter 12: 30788
Chapter 13: 26403
Chapter 14: 35009
Chapter 15: 18089
Chapter 16: 31694
Chapter 17: 32005
Chapter 18: 17238
Chapter 19: 25356
Chapter 20: 14582
Chapter 21: 33713
Chapter 22: 41289
Chapter 23: 22556
Chapter 24: 9868


### Generating Q&A Pairs

Now that we have all the chapters, let's generate Q&A pairs based on that, because they turn unstructured text into clear, learnable instruction formats that improve fine-tuning, reasoning, and evaluation.

Let's generate Q&A pairs with the help of LLM. We will use ollama with a local model, but we can switch to something bigger later on.

First let's create a function called ChatWithOllama. We will build more functions based on this.

In [87]:
from ollama import chat
from ollama import ChatResponse

def ChatWithOllama(systemContent="You are a kindergarten teacher who always use easy words to answer questions.", userContent='Why is the sky blue?', model='qwen3:8b', showThink=True):
    response: ChatResponse = chat(
        model=model, 
        messages=[
            {
                'role': 'system',
                'content': systemContent,
                
            }, 
             {
                'role': 'user',
                'content': userContent,
            }
        ]
    )

    # print(response.message.content)

    result = response.message.content

    if not showThink:
        import re
        result = re.sub(r"<think>.*?</think>\s*", "", result, flags=re.DOTALL)

    return result

def ChatWith(framework="Ollama", systemContent="You are a kindergarten teacher who always use easy words to answer questions.", userContent='Why is the sky blue?', model='qwen3', showThink=True):
    if framework == "Ollama":
        return ChatWithOllama(systemContent=systemContent, userContent=userContent, model=model, showThink=showThink)

And some functions for different purposes

In [84]:
def generalizeWithLLM(text):
    systemContent="""
        You are a thoughtful assistant skilled at abstracting long, detailed texts into their most important general ideas.
    
        Please read the following text and write a generalization that:
        - Extracts the core themes or ideas,
        - Avoids specific numbers, names, or examples unless essential,
        - Focuses on the overall message or logic of the content,
        - Uses simple and clear language, suitable for someone unfamiliar with the subject.
    """

    userContent = f"""
        Please read the following text and write a generalization that:
        - Extracts the core themes or ideas
        - Avoids specific names, numbers, or examples unless essential
        - Focuses on the overall message or logic
        - Uses simple, clear language suitable for someone new to the topic
        
        Text:
        {text}
        

        Return the generalization as bullet points if the ideas are distinct.
    """

    showThink = False

    
    return ChatWith(systemContent=systemContent, userContent=userContent, showThink=showThink)

In [ ]:
print(generalizeWithLLM("hello"))

In [86]:
print(ChatWith(systemContent="""
    """,
         userContent="""
        Please read the following text and write a generalization that:
        - Extracts the core themes or ideas
        - Avoids specific names, numbers, or examples unless essential
        - Focuses on the overall message or logic
        - Uses simple, clear language suitable for someone new to the topic
        
        Text:
        Hello， why are you doing this to me? Don't you know that I cannot survive without you?
        

        Return the generalization as bullet points if the ideas are distinct.
    """,
              showThink = True))

None


Let's split up the content in each chapter, and feed them respectively into the model.

In [52]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

chunks = []
for chapter in chapters:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=300
    )
    chunks += text_splitter.split_text(chapter)
chunks[100]

"between  them had succeeded in closing them up  pretty  \ntight. Besides, I wanted to find a place where the only limit  \nto my trading would be the size of my stake. I didn't have  \nmuch of one, but I didn't expect it to stay little forever. The  \nmain thing at the start was to find a place where I wouldn't  \nhave to worry about getting a square deal. So I went to a  \nNew  York  Stock  Exchange  house  that  had  a  branch  at  \nhome where I knew some of the clerks. They have long  \nsince gone out of business. I wasn't there long, didn't like"